In [ ]:
from pathlib import Path
import numpy as np, torch, matplotlib.pyplot as plt
from trace_comparison.training.imitation import load_records, load_imitation, build_tensors, BOS_VALUE  # <- adjust import path
from utils.notebook_utils import reset_cwd_and_env

reset_cwd_and_env()
BASEPATH = Path.cwd()

SOLVER = "ac_sgl_1"                                   # <- your solver key
DATA   = BASEPATH / "data" / "traces_dataset" / "nonograms" / "base_1000.jsonl"
CKPT   = BASEPATH / "checkpoints" / "imitation" / SOLVER / "02.pt"
# must match the training config so the val split is identical
SEED, TRAIN_FRACTION, MAX_SIZE, PUZZLE_FILTER = 0, 0.8, None, None

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_imitation(CKPT, dev)
records = load_records(DATA, [SOLVER], MAX_SIZE, PUZZLE_FILTER)
data = build_tensors(records, SOLVER, model.max_steps)

n = len(records)
perm = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
n_train = max(1, min(int(TRAIN_FRACTION * n), n - 1))
val_idx = perm[n_train:]
print(f"{n} puzzles, {len(val_idx)} val, grid {records[0].grid_shape}, max_steps {model.max_steps}")

In [ ]:
clues, cmask, trace, smask = (t[val_idx].to(dev) for t in data)
prev = torch.cat([torch.full_like(trace[:, :1], BOS_VALUE), trace[:, :-1]], 1)
with torch.no_grad():
    pred = model.eval()(clues, cmask, prev)

valid = smask[:, :, None, None].expand_as(trace)          # real (unpadded) steps
changed = (trace != prev) & valid                          # cells that differ from the previous step
sq_model, sq_copy = (pred - trace).square(), (prev - trace).square()

print(f"all cells      | model {sq_model[valid].mean():.5f} | copy-prev {sq_copy[valid].mean():.5f}")
print(f"changed cells  | model {sq_model[changed].mean():.5f} | copy-prev {sq_copy[changed].mean():.5f}")
print(f"fraction of cells that change per step: {changed.sum().item() / valid.sum().item():.3%}")

In [ ]:
k = 3
prev2 = prev.clone()
prev2[:, k + 1:] = torch.rand_like(prev2[:, k + 1:])       # scramble inputs of steps > k
with torch.no_grad():
    pred2 = model(clues, cmask, prev2)
print("max change in outputs of steps <= k:", (pred2[:, :k + 1] - pred[:, :k + 1]).abs().max().item())
# should be ~0 (1e-6 or smaller); anything larger means future information is leaking

In [ ]:
@torch.no_grad()
def rollout_errors(i):
    rec = records[i]
    true = rec.runs[SOLVER][: model.max_steps]
    gen = model.generate(rec.clues)[: len(true)]
    return gen, true

# aggregate over a handful of val puzzles
final_err, const_err = [], []
for i in val_idx[:20].tolist():
    gen, true = rollout_errors(i)
    final_err.append(((gen[-1] - true[-1]) ** 2).mean())
    const_err.append(((BOS_VALUE - true[-1]) ** 2).mean())   # "everything still unknown" baseline
print(f"final-grid MSE: model {np.mean(final_err):.4f} | all-unknown baseline {np.mean(const_err):.4f}")

# one puzzle in detail
gen, true = rollout_errors(val_idx[0].item())
T = len(true)
steps = sorted({0, T // 4, T // 2, T - 1})
fig, ax = plt.subplots(2, len(steps), figsize=(3 * len(steps), 6), squeeze=False)
for j, s in enumerate(steps):
    ax[0, j].imshow(true[s], vmin=0, vmax=1, cmap="gray_r"); ax[0, j].set_title(f"true, step {s}")
    ax[1, j].imshow(gen[s], vmin=0, vmax=1, cmap="gray_r"); ax[1, j].set_title(f"generated, step {s}")
for a in ax.ravel(): a.axis("off")
plt.show()

plt.plot(((gen - true) ** 2).mean((1, 2))); plt.xlabel("step"); plt.ylabel("MSE vs true trace")
plt.title("rollout error over time"); plt.show()

In [ ]:
clues, cmask, trace, smask = (t[val_idx].to(dev) for t in data)
prev = torch.cat([torch.full_like(trace[:, :1], BOS_VALUE), trace[:, :-1]], 1)
w = smask.float()

def per_step(err):                       # err: (N, T) -> masked mean per step
    return (err * w).sum(0) / w.sum(0).clamp(min=1)

@torch.no_grad()
def model_mse(c, cm):
    return per_step((model.eval()(c, cm, prev) - trace).square().mean((2, 3)))

sh = torch.randperm(len(clues), generator=torch.Generator().manual_seed(0)).to(dev)
true_c, shuf_c = model_mse(clues, cmask), model_mse(clues[sh], cmask[sh])
copy = per_step((prev - trace).square().mean((2, 3)))

print("step | model (true clues) | model (shuffled clues) | copy-prev")
for t in range(trace.shape[1]):
    if w[:, t].sum() > 0:
        print(f"{t:4d} | {true_c[t]:.5f} | {shuf_c[t]:.5f} | {copy[t]:.5f}")
print(f"overall: true {(true_c*w.sum(0)).sum()/w.sum():.5f} | shuffled {(shuf_c*w.sum(0)).sum()/w.sum():.5f}")